# 09 — FlightRescue AI Final Inference Pipeline

This is the **final integration notebook** in the research/notebook phase. It combines:

1. flight-level **any-disruption risk**;
2. flight-level **severe-disruption risk**;
3. historical **similar-event retrieval**;
4. historical cancellation/severe-disruption evidence;
5. historical operational-recovery evidence; and
6. a simple confidence/risk layer suitable for the later API and UI.

The notebook intentionally reuses the modeling design from Notebook 06 and the event representation from Notebook 08. It does not introduce a new scientific experiment.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import (
    average_precision_score, roc_auc_score, f1_score,
    precision_score, recall_score, balanced_accuracy_score
)
from sklearn.metrics.pairwise import cosine_similarity

cwd = Path.cwd().resolve()
if (cwd / 'data').exists():
    ROOT = cwd
elif (cwd.parent / 'data').exists():
    ROOT = cwd.parent
else:
    raise FileNotFoundError(f'Cannot locate project root from {cwd}')

FEATURE_FILE = ROOT / 'data/processed/ogg_model_features_v1.csv.gz'
EVENT_FILE = ROOT / 'data/processed/ogg_weather_event_recovery_2020_2025.csv'
SIM_INDEX_FILE = ROOT / 'data/processed/ogg_event_similarity_index_2020_2025.csv'
ARTIFACT_DIR = ROOT / 'models'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

for p in [FEATURE_FILE, EVENT_FILE, SIM_INDEX_FILE]:
    print(p, 'exists=', p.exists())


## 1. Load model and historical-event data


In [ ]:
df = pd.read_csv(FEATURE_FILE, low_memory=False)
events = pd.read_csv(EVENT_FILE, low_memory=False)
sim_index = pd.read_csv(SIM_INDEX_FILE, low_memory=False)

for c in ['start_dt','end_dt','recovery_dt']:
    if c in events.columns:
        events[c] = pd.to_datetime(events[c], errors='coerce')

print('Flight feature table:', df.shape)
print('Historical event table:', events.shape)
print('Similarity index:', sim_index.shape)


## 2. Recreate the temporal split and binary targets from Notebook 06


In [ ]:
if 'split' not in df.columns:
    raise KeyError("Expected 'split' column from Notebook 04.")

train_df = df[df['split'].eq('train')].copy()
val_df = df[df['split'].eq('validation')].copy()
test_df = df[df['split'].eq('test')].copy()

for part in [train_df, val_df, test_df]:
    part['target_any_disruption'] = (part['disruption_class'] != 'normal').astype(int)
    part['target_severe'] = part['disruption_class'].isin(['severe_delay','cancelled']).astype(int)

display(pd.DataFrame({
    'split':['train','validation','test'],
    'rows':[len(train_df),len(val_df),len(test_df)],
    'any_disruption_pct':[100*train_df['target_any_disruption'].mean(),100*val_df['target_any_disruption'].mean(),100*test_df['target_any_disruption'].mean()],
    'severe_pct':[100*train_df['target_severe'].mean(),100*val_df['target_severe'].mean(),100*test_df['target_severe'].mean()],
}).round(3))


## 3. Leakage-safe features and preprocessing


In [ ]:
exclude = {
    'disruption_class','target_any_disruption','target_severe','split',
    'FlightDate','ogg_sched_dt','weather_dt',
    'Cancelled','CancellationCode','Diverted','DepTime','ArrTime',
    'DepDelay','ArrDelay','DepDelayMinutes','ArrDelayMinutes',
    'CarrierDelay','WeatherDelay','NASDelay','SecurityDelay','LateAircraftDelay',
}
feature_cols = [c for c in train_df.columns if c not in exclude and not train_df[c].isna().all()]
numeric_cols = [c for c in feature_cols if pd.api.types.is_numeric_dtype(train_df[c])]
categorical_cols = [c for c in feature_cols if c not in numeric_cols]

print('Features:', len(feature_cols))
print('Numeric:', len(numeric_cols), 'Categorical:', len(categorical_cols))

linear_preprocess = ColumnTransformer([
    ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), numeric_cols),
    ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))]), categorical_cols),
])

tree_preprocess = ColumnTransformer([
    ('num', SimpleImputer(strategy='median'), numeric_cols),
    ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('ordinal', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))]), categorical_cols),
])

def make_logreg():
    return Pipeline([
        ('preprocess', linear_preprocess),
        ('model', LogisticRegression(C=1.0, max_iter=1000, class_weight='balanced', solver='lbfgs', tol=1e-4)),
    ])

def make_histgb():
    return Pipeline([
        ('preprocess', tree_preprocess),
        ('model', HistGradientBoostingClassifier(
            learning_rate=0.08, max_iter=250, max_leaf_nodes=31,
            min_samples_leaf=30, l2_regularization=1.0,
            class_weight='balanced', random_state=42
        )),
    ])


## 4. Validation-based model and threshold selection
The same principle as Notebook 06 is retained: select using 2024 validation only; 2025 remains untouched until final reporting.


In [ ]:
def threshold_table(y_true, prob):
    rows = []
    for t in np.arange(0.05, 0.951, 0.025):
        pred = (prob >= t).astype(int)
        rows.append({
            'threshold': t,
            'precision': precision_score(y_true, pred, zero_division=0),
            'recall': recall_score(y_true, pred, zero_division=0),
            'f1': f1_score(y_true, pred, zero_division=0),
            'balanced_accuracy': balanced_accuracy_score(y_true, pred),
        })
    return pd.DataFrame(rows)

def fit_select(target):
    X_train, X_val = train_df[feature_cols], val_df[feature_cols]
    y_train, y_val = train_df[target], val_df[target]
    candidates = {'logistic_regression': make_logreg(), 'hist_gradient_boosting': make_histgb()}
    rows, trained = [], {}
    for name, model in candidates.items():
        print('Training', target, name)
        model.fit(X_train, y_train)
        prob = model.predict_proba(X_val)[:,1]
        tt = threshold_table(y_val, prob)
        best = tt.sort_values(['f1','recall'], ascending=False).iloc[0]
        rows.append({
            'model': name,
            'pr_auc': average_precision_score(y_val, prob),
            'roc_auc': roc_auc_score(y_val, prob),
            'threshold': float(best['threshold']),
            'f1': float(best['f1']),
            'precision': float(best['precision']),
            'recall': float(best['recall']),
        })
        trained[name] = model
    res = pd.DataFrame(rows).sort_values(['pr_auc','f1'], ascending=False).reset_index(drop=True)
    winner = res.iloc[0]
    return trained[winner['model']], float(winner['threshold']), res

any_model, any_threshold, any_validation = fit_select('target_any_disruption')
severe_model, severe_threshold, severe_validation = fit_select('target_severe')

print('Any-disruption validation:')
display(any_validation.round(4))
print('Severe-disruption validation:')
display(severe_validation.round(4))
print('Selected thresholds:', {'any': any_threshold, 'severe': severe_threshold})


## 5. Historical-analog representation from Notebook 08


In [ ]:
z_cols = [c for c in sim_index.columns if c.startswith('z__')]
if not z_cols:
    raise ValueError('Similarity index contains no standardized z__ features. Re-run Notebook 08.')

Z = sim_index[z_cols].apply(pd.to_numeric, errors='coerce').fillna(0).to_numpy()

def event_type_set(value):
    if pd.isna(value): return set()
    return {x.strip().lower() for x in str(value).split('|') if x.strip()}

def jaccard(a, b):
    a, b = event_type_set(a), event_type_set(b)
    union = a | b
    return len(a & b)/len(union) if union else 1.0

def retrieve_event_analogs(event_id, k=5, numeric_weight=0.80, type_weight=0.20):
    idxs = sim_index.index[sim_index['event_id'].eq(event_id)].tolist()
    if not idxs:
        raise KeyError(f'Unknown historical event_id: {event_id}')
    q = idxs[0]
    numeric_sim = cosine_similarity(Z[q:q+1], Z).ravel()
    qtype = sim_index.loc[q, 'event_types'] if 'event_types' in sim_index else ''
    type_sim = np.array([jaccard(qtype, x) for x in sim_index.get('event_types', pd.Series(['']*len(sim_index)))])
    score = numeric_weight*numeric_sim + type_weight*type_sim
    score[q] = -np.inf
    top = np.argsort(score)[::-1][:k]
    cols = [c for c in ['event_id','start_dt','end_dt','event_types','event_cancel_rate','event_severe_rate','recovery_hours_after_event'] if c in sim_index.columns]
    out = sim_index.loc[top, cols].copy()
    out.insert(1, 'similarity_score', score[top])
    return out.reset_index(drop=True)


## 6. Risk and confidence helpers


In [ ]:
def risk_label(prob, threshold):
    if prob < 0.50*threshold:
        return 'LOW'
    if prob < threshold:
        return 'MODERATE'
    if prob < min(1.0, threshold + 0.20):
        return 'HIGH'
    return 'VERY HIGH'

def confidence_label(analogs):
    if analogs is None or len(analogs) == 0:
        return 'LOW'
    med = analogs['similarity_score'].median()
    rec = analogs['recovery_hours_after_event'].dropna() if 'recovery_hours_after_event' in analogs else pd.Series(dtype=float)
    spread = rec.max()-rec.min() if len(rec) >= 2 else np.inf
    if med >= 0.80 and spread <= 12:
        return 'HIGH'
    if med >= 0.60 and spread <= 24:
        return 'MODERATE'
    return 'LOW'


## 7. Final reusable inference function

For now the function accepts a feature dictionary/Series plus an optional historical `event_id`. In the production app, the feature dictionary will be assembled automatically from flight schedule + live/forecast weather, and the event query will be built directly from current conditions rather than requiring a historical ID.


In [ ]:
def predict_flightrescue(flight_features, historical_event_id=None, k_analogs=5):
    if isinstance(flight_features, pd.Series):
        row = flight_features.to_dict()
    else:
        row = dict(flight_features)

    X_one = pd.DataFrame([{c: row.get(c, np.nan) for c in feature_cols}])
    any_prob = float(any_model.predict_proba(X_one)[:,1][0])
    severe_prob = float(severe_model.predict_proba(X_one)[:,1][0])

    analogs = retrieve_event_analogs(historical_event_id, k=k_analogs) if historical_event_id else pd.DataFrame()
    recovered = analogs['recovery_hours_after_event'].dropna() if len(analogs) and 'recovery_hours_after_event' in analogs else pd.Series(dtype=float)
    cancel = analogs['event_cancel_rate'].dropna() if len(analogs) and 'event_cancel_rate' in analogs else pd.Series(dtype=float)
    severe_hist = analogs['event_severe_rate'].dropna() if len(analogs) and 'event_severe_rate' in analogs else pd.Series(dtype=float)

    result = {
        'disruption_probability': any_prob,
        'disruption_risk': risk_label(any_prob, any_threshold),
        'severe_disruption_probability': severe_prob,
        'severe_disruption_risk': risk_label(severe_prob, severe_threshold),
        'any_disruption_threshold': any_threshold,
        'severe_disruption_threshold': severe_threshold,
        'historical_event_id': historical_event_id,
        'analogs_used': int(len(analogs)),
        'median_analog_similarity': float(analogs['similarity_score'].median()) if len(analogs) else None,
        'historical_median_cancel_rate': float(cancel.median()) if len(cancel) else None,
        'historical_median_severe_rate': float(severe_hist.median()) if len(severe_hist) else None,
        'estimated_recovery_hours': float(recovered.median()) if len(recovered) else None,
        'recovery_range_hours': [float(recovered.min()), float(recovered.max())] if len(recovered) else None,
        'confidence': confidence_label(analogs),
        'similar_events': analogs.to_dict(orient='records') if len(analogs) else [],
    }
    return result


## 8. End-to-end demonstration on a 2025 test flight


In [ ]:
# Pick a 2025 test flight that was actually disrupted, if available.
demo_candidates = test_df[test_df['target_any_disruption'].eq(1)]
demo_flight = demo_candidates.iloc[0] if len(demo_candidates) else test_df.iloc[0]

# Use the most operationally severe 2025 historical event as a demonstration event context.
if 'year' in events.columns:
    ep = events[events['year'].eq(2025)].copy()
else:
    ep = events.copy()
if 'event_severe_rate' in ep.columns and ep['event_severe_rate'].notna().any():
    demo_event_id = ep.sort_values('event_severe_rate', ascending=False).iloc[0]['event_id']
else:
    demo_event_id = events.iloc[-1]['event_id']

demo_result = predict_flightrescue(demo_flight, historical_event_id=demo_event_id, k_analogs=5)
print(json.dumps({k:v for k,v in demo_result.items() if k != 'similar_events'}, indent=2, default=str))
display(pd.DataFrame(demo_result['similar_events']).round(3))


## 9. Final untouched 2025 probability sanity check


In [ ]:
X_test = test_df[feature_cols]
for target, model, threshold in [
    ('target_any_disruption', any_model, any_threshold),
    ('target_severe', severe_model, severe_threshold),
]:
    y = test_df[target]
    prob = model.predict_proba(X_test)[:,1]
    pred = (prob >= threshold).astype(int)
    print('\n', target)
    print({
        'roc_auc': round(roc_auc_score(y, prob),4),
        'pr_auc': round(average_precision_score(y, prob),4),
        'threshold': round(threshold,3),
        'precision': round(precision_score(y,pred,zero_division=0),4),
        'recall': round(recall_score(y,pred,zero_division=0),4),
        'f1': round(f1_score(y,pred,zero_division=0),4),
        'balanced_accuracy': round(balanced_accuracy_score(y,pred),4),
    })


## 10. Save integration metadata for the app layer


In [ ]:
metadata = {
    'feature_columns': feature_cols,
    'numeric_columns': numeric_cols,
    'categorical_columns': categorical_cols,
    'any_disruption_threshold': any_threshold,
    'severe_disruption_threshold': severe_threshold,
    'historical_similarity_features': [c.replace('z__','',1) for c in z_cols],
    'notes': {
        'recovery': 'BTS-derived operational recovery proxy, not an official FAA reopening time',
        'validation_design': 'Train 2020-2023, validation 2024, test 2025',
    },
}
meta_path = ARTIFACT_DIR / 'flightrescue_inference_metadata.json'
with open(meta_path, 'w') as f:
    json.dump(metadata, f, indent=2)
print('Saved:', meta_path)


# Notebook phase complete

After this notebook, development moves out of exploratory notebooks and into application code:

- `src/` reusable inference modules;
- model/artifact persistence;
- current/forecast weather ingestion;
- API/service layer; and
- GitHub Pages UI/UX.

Notebook 09 is therefore the bridge between the research pipeline and the deployable FlightRescue AI product.
